# Calculating a hydration free energy with pontibus

This notebook briefly demonstrates how to create a ``Transformation`` to calculate the hydration free energy of a solute (benzene) in a box of water.

In [1]:
# Some imports you will need later
from openff.toolkit import Molecule
from openff.units import unit
import openfe
from pontibus.protocols.solvation import ASFEProtocol
from pontibus.components import ExtendedSolventComponent
from pontibus.protocols.solvation.settings import PackmolSolvationSettings

## Defining the system's molecules

### The solute

We start by creating the solute (benzene) and assigning partial charges. Here we use AshGC charges, but you can use other charge models if needed.

How you generate solute molecules may difer depending on your needs. Here we start with an `openff.toolkit.Molecule` generated from a smiles string, but you could load in an SDF into an RDKit Molecule and then convert it to an OpenFF Molecule or skip the OpenFF Molecule step altogether if you already have partial charges!

In all cases, you will need to end up with an `openfe.SmallMoleculeComponent.`

In [2]:
# Start by creating an OpenFF Molecule from smiles, so we also need to generate a conformer
solute_offmol = Molecule.from_smiles("c1ccccc1")
solute_offmol.generate_conformers(n_conformers=1)

In [3]:
# Now assign partial charges
solute_offmol.assign_partial_charges(partial_charge_method="openff-gnn-am1bcc-1.0.0.pt")

In [4]:
# Now we create the openfe SmallMoleculeComponent
solute_smc = openfe.SmallMoleculeComponent.from_openff(solute_offmol)

### The solvent

We also create a `openfe.SmallMoleculeComponent` for the solvent (water). In this case we don't need to assign partial charges because we will be using the force field's charges for water instead.

In [5]:
solvent_offmol = Molecule.from_smiles('O')
solvent_offmol.generate_conformers(n_conformers=1)
solvent_smc = openfe.SmallMoleculeComponent.from_openff(solvent_offmol)

## Creating the ChemicalSystems defining the end states

In `pontibus` and `openfe`, absolute hydration and solvation free energies are defined as a solute in solvent (state A) being transformed to just the solvent (state B).

Unlike `openfe`, `pontibus` preferrentially uses a modified version of `SolventComponent` called `ExtendedSolventComponent`. This allows users to define extra properties for the solvent that will be added, including the solvent conformer and optionally partial charges.

In [6]:
# We start by creating the ExtendedSolventComponent from the solvent_smc
solvent_component = ExtendedSolventComponent(solvent_molecule=solvent_smc)

In [7]:
# Now we create the ChemicalSystems for state A and B
stateA = openfe.ChemicalSystem({'solute': solute_smc, 'solvent': solvent_component}, name="benzene")
stateB = openfe.ChemicalSystem({'solvent': solvent_component}, name="water")

## Creating the settings

Next we create settings for our simulation. For the most part we use the default settings, but there are a few tweaks we need to do.

In [8]:
settings = ASFEProtocol.default_settings()

In [9]:
# Non default settings we need to do

# For the water solvation, we set a dodecahedron around the solute with
# a minimum of 1.5 nm solvent padding. In our tests this works for freesolv.
# In some cases, especially with ligands that can unfold, you may need to
# bump this up.
settings.solvation_settings = PackmolSolvationSettings(
    box_shape="dodecahedron",
    assign_solvent_charges=False,
    solvent_padding=1.5 * unit.nanometer,
)

# The default force field is somewhat conservative, we set it here to use openff 2.3.0 which is made for AshGC charges
settings.solvent_forcefield_settings.forcefields = [
    # To use a custom force field, just pass an OFFXML string
    # just like you would to openff.toolkit.ForceField
    "openff-2.3.0.offxml",
    "tip3p.offxml",
]
settings.vacuum_forcefield_settings.forcefields = [
    "openff-2.3.0.offxml",  # as above
]

# Finally we set the number of repeats to 1, so that you can run multiple repeats independently
settings.protocol_repeats = 1

In [10]:
# Here are some default settings you may need to be aware of

# This defines the compute platform to use, you'll usually
# want to just set it to CUDA so you can fail fast if the GPU isn't available
settings.vacuum_engine_settings.compute_platform = "CUDA"
settings.solvent_engine_settings.compute_platform = "CUDA"

# Below are the default lambda & replica settings, so you don't have to
# actually set them, but if you want to change things, you can alter them
# by defining them this way. Note: you have to update n_replica to match
# the number of lambda windows (and all lambda window lists must be of the same length).
settings.lambda_settings.lambda_elec = [
    0.0, 0.25, 0.5, 0.75, 1.0,
    1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0,
]
settings.lambda_settings.lambda_vdw = [
    0.0, 0.0, 0.0, 0.0, 0.0,
    0.12, 0.24, 0.36, 0.48, 0.6, 0.7, 0.77, 0.85, 1.0,
]
settings.lambda_settings.lambda_restraints = [
    0.0, 0.0, 0.0, 0.0, 0.0,
    0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,
]
settings.vacuum_simulation_settings.n_replicas = 14
settings.solvent_simulation_settings.n_replicas = 14

# Below are the default simulation lengths we use in Pontibus,
# so you don't need to set them. However, you can do so manually
# This is the pre-alchemical equilibration lengths
# NVT equilibration -> NPT equilibration -> NPT "production" (more equilibration)
# In vacuum, we set the NVT equilibration to None since it's all gas phase
settings.solvent_equil_simulation_settings.equilibration_length_nvt = 0.5 * unit.nanosecond
settings.solvent_equil_simulation_settings.equilibration_length = 0.5 * unit.nanosecond
settings.solvent_equil_simulation_settings.production_length = 9.5 * unit.nanosecond
settings.vacuum_equil_simulation_settings.equilibration_length_nvt = None
settings.vacuum_equil_simulation_settings.equilibration_length = 0.2 * unit.nanosecond
settings.vacuum_equil_simulation_settings.production_length = 0.5 * unit.nanosecond
# This is the alchemical equilibration length
settings.solvent_simulation_settings.equilibration_length = 1.0 * unit.nanosecond
settings.vacuum_simulation_settings.equilibration_length = 0.5 * unit.nanosecond
# This is the alchemical production length
settings.solvent_simulation_settings.production_length = 10.0 * unit.nanosecond
settings.vacuum_simulation_settings.production_length = 2.0 * unit.nanosecond

## Creating the Protocol

Similar to all other `openfe` Protocols, you can create the ASFEProtocol by passing in the settings.

In [11]:
protocol = ASFEProtocol(settings=settings)

## Creating the Transformation

You can then create ASFE Transformations by passing in the `ChemicalSystem` end states.

**Note:** the mapping is `None` because this is an absolute transformation so we don't need to interpolate atoms between the end states.

In [12]:
transformation = openfe.Transformation(stateA=stateA, stateB=stateB, mapping=None, protocol=protocol, name=stateA.name)

## Running the Transformation locally

Similar to in `openfe` you can run the transformation locally using the `openfe quickrun` CLI command.

First you need to write the Transformation to file:

In [13]:
transformation.to_json('benzene_asfe_transformation.json')

Then you can call quickrun in the following manner:

```
openfe quickrun benzene_asfe_transformation.json -o results/benzene_asfe.json -d results/benzene_asfe
```